### Assignment: Quantum Primitives 2
Apply the non-unitary matrix $ \begin{bmatrix} 1 & 0 \\ 0 & 0 \end{bmatrix}$

on a 1-qubit quantum state which has a 0.3 probability being measured in the zero state |0〉.





Since the operator is not unitary we can not apply it directly on the one qubit quantum state we have to use LCU method to be able to perform this operation:

We can use a simple LCU decomposition which holds for any projector:
An LCU decomposition that holds for any projector is:
                               

$$ |\phi\rangle \langle \phi| = \frac{1}{2} I + \frac{1}{2} \left( 2 |\phi\rangle \langle \phi| - I \right) $$


we can now use this LCU  to write the implementation
$$ |0\rangle \langle 0| = \begin{bmatrix} 1 & 0 \\ 0 & 0 \end{bmatrix}. $$
Note that the second term in our LCU simplifies to a Pauli Z operation.
We can now construct a full LCU circuit and verify that |0><0|  is block-encoded in the top left block of the matrix.

In [3]:
The coefficients are going to be [1/2 1/2]
And the corresponding controllers here are Z, I. 
We will also prepare the state  $ |\psi\rangle $ with the right probabilities of [0.3 0.7]
lambda here is one.


SyntaxError: invalid syntax (2772697371.py, line 1)

In [4]:
Now we are going to perform those steps:


SyntaxError: invalid syntax (2630751485.py, line 1)

In [5]:
from classiq import *
import numpy as np

@qfunc
def lcu_controllers(controller: QNum, psi: QArray[QBit]):
    # Apply the identity operation when controller == 0
    control(ctrl=controller == 0, operand=lambda: apply_to_all(IDENTITY, psi))
    # Apply the Pauli-Z operation when controller == 1
    control(ctrl=controller == 1, operand=lambda: apply_to_all(Z, psi))

@qfunc
def main(controller: Output[QNum], psi:  Output[QArray[QBit]]):
    
    # Define the error bound and probability distribution
    error_bound = 0.01
    controller_probabilities = [0.5, 0.5]  # Probabilities for I and Z respectively
    
    # Allocate 1 qubit for the controller (used to select the unitary)
    allocate(1, controller)
    prepare_state(probabilities=[ 0.3, 0.7], bound=0.01, out=psi)

    

    # Execute the Within-Apply function
    within_apply(
        compute=lambda: inplace_prepare_state(
            probabilities=controller_probabilities, bound=error_bound, target=controller
        ),
        action=lambda: lcu_controllers(controller, psi),
    )
from classiq.execution import ExecutionPreferences

# Create the quantum model
quantum_model = create_model(main)

# Set execution preferences
execution_preferences = ExecutionPreferences(
    num_shots=1000,
    job_name="classiq_101_execute",
    random_seed=767
)

# Apply execution preferences to the quantum model
quantum_model_with_execution_preferences = set_execution_preferences(
    quantum_model,
    execution_preferences
)

# Synthesize the quantum program
quantum_program = synthesize(quantum_model_with_execution_preferences)

# Show the quantum program (assuming show() method displays or stores the program)
#show(quantum_program)

# Execute the quantum program
job = execute(quantum_program)

# Retrieve and print results
# Retrieve and print results
results = job.result()[0].value
print("Parsed Counts:", results.parsed_counts)
print("Raw Counts:", results.counts)


show(quantum_program)

Parsed Counts: [{'controller': 1.0, 'psi': 1.0}: 702, {'controller': 0.0, 'psi': 0.0}: 298]
Raw Counts: {'00': 298, '11': 702}
Opening: https://platform.classiq.io/circuit/57937161-5e61-47dd-b958-e3abd654febf?version=0.43.2


Parsed Counts:
{'controller': 1.0, 'psi': 1.0}: 702: This indicates that the state where controller is 1.0 and psi is 1.0 (likely corresponding 
to qubit states or computational basis states) occurred 702 times.
{'controller': 0.0, 'psi': 0.0}: 298: This indicates that the state where controller is 0.0 and psi is 0.0 occurred 298 times.
Raw Counts:
{'00': 298, '11': 702}: This shows the raw measurement outcomes in terms of qubit states. 00 occurred 298 times, and 11 occurred 702 times.
We knew that $$ \langle 0 | \text{PREP}^\dagger \cdot \text{SEL} \cdot \text{PREP} | 0 \rangle |\psi\rangle = A /\lambda |\psi\rangle. $$
    
So having as a result {'00': 298, '11': 702}  a probability of approximatly 0.3 to see psi the state |0> seeems logical since the LCU once 
applied to a non unitary operator will give us  $ |\psi\rangle $ back.